In [ ]:
import pandas as pd
import numpy as np

In [ ]:
up = r"C:\Users\karol\Desktop\Data_Science\ISA\jdszr25-grupa-1\Datasets"

meta = pd.read_csv(up + 'metadata.csv')
meta_eui = meta[meta['eui'].notna()].copy()
valid_ids = set(meta_eui['building_id'])
print("Buildings with EUI:", len(valid_ids))

meter_files = {
    'electricity': 'electricity.csv',
    'gas': 'gas.csv',
    'solar': 'solar.csv',
    'steam': 'steam.csv',
}

long_frames = []
for meter_name, fname in meter_files.items():
    df = pd.read_csv(up + fname)
    cols = [c for c in df.columns if c in valid_ids]
    if not cols:
        continue
    sub = df[['timestamp'] + cols]
    del df
    melted = sub.melt(id_vars='timestamp', var_name='building_id', value_name='reading')
    del sub
    melted['meter'] = meter_name
    melted = melted.dropna(subset=['reading'])
    melted['reading'] = melted['reading'].astype('float32')
    melted['building_id'] = melted['building_id'].astype('category')
    melted['meter'] = melted['meter'].astype('category')
    long_frames.append(melted)
    print(meter_name, "rows:", len(melted))

long_df = pd.concat(long_frames, ignore_index=True)
del long_frames
print("Total long rows:", len(long_df))

bid_to_site = meta_eui.set_index('building_id')['site_id'].to_dict()
long_df['site_id'] = long_df['building_id'].map(bid_to_site).astype('category')

weather = pd.read_csv(up + 'weather.csv')
for c in ['airTemperature','cloudCoverage','dewTemperature','precipDepth1HR','precipDepth6HR','seaLvlPressure','windDirection','windSpeed']:
    weather[c] = weather[c].astype('float32')

merged = long_df.merge(weather, on=['timestamp', 'site_id'], how='left')
del long_df, weather
print("After weather merge:", merged.shape)

meta_cols = ['building_id','site_id','primaryspaceusage','sub_primaryspaceusage','sqm','sqft',
             'lat','lng','timezone','yearbuilt','eui','site_eui','source_eui']
meta_small = meta_eui[meta_cols].copy()
for c in ['sqm','sqft','lat','lng','yearbuilt','eui','site_eui','source_eui']:
    meta_small[c] = pd.to_numeric(meta_small[c].astype(str).str.replace(',', '', regex=False), errors='coerce').astype('float32')
for c in ['primaryspaceusage','sub_primaryspaceusage','timezone']:
    meta_small[c] = meta_small[c].astype('category')

merged = merged.merge(meta_small, on=['building_id','site_id'], how='left')
print("Final shape:", merged.shape)

merged['timestamp'] = pd.to_datetime(merged['timestamp'])




In [ ]:
out_path = './merged_dataset.parquet'
merged.to_parquet(out_path, index=False, compression='snappy')
print("Saved to", out_path)

In [2]:
!pip install polars
import polars as pl

df = pl.scan_parquet('merged_dataset.parquet').head(5).collect()
print(df)

   ---------------------------------------- 0.0/865.2 kB ? eta -:--:--
   ----------------------------------- --- 786.4/865.2 kB 19.5 MB/s eta 0:00:01
   ---------------------------------------- 865.2/865.2 kB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/53.7 MB ? eta -:--:--
   ------------- -------------------------- 18.4/53.7 MB 86.6 MB/s eta 0:00:01
   ------------------- -------------------- 26.7/53.7 MB 65.4 MB/s eta 0:00:01
   ------------------------------- -------- 41.9/53.7 MB 65.1 MB/s eta 0:00:01
   ---------------------------------------  53.5/53.7 MB 70.0 MB/s eta 0:00:01
   ---------------------------------------- 53.7/53.7 MB 58.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ----------------------------------

In [1]:
import polars as pl

# tworzy plan zapytania, NIE czyta jeszcze danych
lf = pl.scan_parquet('merged_dataset.parquet')

# nazwy kolumn i typy - to jest "darmowe", polars czyta tylko metadane pliku
print(lf.columns)
print(lf.schema)

# podgląd danych - dopiero .collect() faktycznie wykonuje odczyt
lf.head(5).collect()

['timestamp', 'building_id', 'reading', 'meter', 'site_id', 'airTemperature', 'cloudCoverage', 'dewTemperature', 'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure', 'windDirection', 'windSpeed', 'primaryspaceusage', 'sub_primaryspaceusage', 'sqm', 'sqft', 'lat', 'lng', 'timezone', 'yearbuilt', 'eui', 'site_eui', 'source_eui']
Schema({'timestamp': Datetime(time_unit='us', time_zone=None), 'building_id': String, 'reading': Float32, 'meter': String, 'site_id': String, 'airTemperature': Float32, 'cloudCoverage': Float32, 'dewTemperature': Float32, 'precipDepth1HR': Float32, 'precipDepth6HR': Float32, 'seaLvlPressure': Float32, 'windDirection': Float32, 'windSpeed': Float32, 'primaryspaceusage': Categorical, 'sub_primaryspaceusage': Categorical, 'sqm': Float32, 'sqft': Float32, 'lat': Float32, 'lng': Float32, 'timezone': Categorical, 'yearbuilt': Float32, 'eui': Float32, 'site_eui': Float32, 'source_eui': Float32})


C:\Users\karol\AppData\Local\Temp\ipykernel_29300\384996162.py:7: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(lf.columns)
C:\Users\karol\AppData\Local\Temp\ipykernel_29300\384996162.py:8: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(lf.schema)


timestamp,building_id,reading,meter,site_id,airTemperature,cloudCoverage,dewTemperature,precipDepth1HR,precipDepth6HR,seaLvlPressure,windDirection,windSpeed,primaryspaceusage,sub_primaryspaceusage,sqm,sqft,lat,lng,timezone,yearbuilt,eui,site_eui,source_eui
datetime[μs],str,f32,str,str,f32,f32,f32,f32,f32,f32,f32,f32,cat,cat,f32,f32,f32,f32,cat,f32,f32,f32,f32
2016-01-01 00:00:00,"""Panther_parking_Lorriane""",0.0,"""electricity""","""Panther""",19.4,null,19.4,0.0,null,null,0.0,0.0,"""Parking""","""Parking Garage""",36012.699219,387638.0,28.517689,-81.379036,"""US/Eastern""",2008.0,8.0,null,null
2016-01-01 01:00:00,"""Panther_parking_Lorriane""",0.0,"""electricity""","""Panther""",21.1,6.0,21.1,-1.0,null,1019.400024,0.0,0.0,"""Parking""","""Parking Garage""",36012.699219,387638.0,28.517689,-81.379036,"""US/Eastern""",2008.0,8.0,null,null
2016-01-01 02:00:00,"""Panther_parking_Lorriane""",0.0,"""electricity""","""Panther""",21.1,null,21.1,0.0,null,1018.799988,210.0,1.5,"""Parking""","""Parking Garage""",36012.699219,387638.0,28.517689,-81.379036,"""US/Eastern""",2008.0,8.0,null,null
2016-01-01 03:00:00,"""Panther_parking_Lorriane""",0.0,"""electricity""","""Panther""",20.6,null,20.0,0.0,null,1018.099976,0.0,0.0,"""Parking""","""Parking Garage""",36012.699219,387638.0,28.517689,-81.379036,"""US/Eastern""",2008.0,8.0,null,null
2016-01-01 04:00:00,"""Panther_parking_Lorriane""",0.0,"""electricity""","""Panther""",21.1,null,20.6,0.0,null,1019.0,290.0,1.5,"""Parking""","""Parking Garage""",36012.699219,387638.0,28.517689,-81.379036,"""US/Eastern""",2008.0,8.0,null,null


In [2]:
row_count = lf.select(pl.len()).collect().item()
print(row_count)

6736885
